# Homework 4 — Evaluation

LLM Zoomcamp 2026 · Cohort

HW2 gave us three ways to search the course lessons — keyword, vector, and hybrid — but left an open question: which one is actually best? This homework answers that with numbers instead of intuition.

We generate a ground-truth dataset (questions with known correct answers), then measure **Hit Rate** and **MRR** for each search method against it.

**Knowledge base:** same 72 lesson markdown pages from commit `8c1834d`, chunked exactly as in HW2 (`size=2000, step=1000` → 295 chunks).

## Setup

We reuse HW2's helper files directly (`embedder.py`, `download.py`, the downloaded ONNX model) via `sys.path`, instead of copying the ~90MB model directory again.

New dependencies for this module (LLM calls with structured output, CSV handling):

```bash
uv add openai pydantic python-dotenv pandas
```

In [3]:
import sys
import json
import os
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import minsearch
from tqdm.auto import tqdm
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import OpenAI



from gitsource import GithubRepositoryDataReader, chunk_documents

# embedder.py and the downloaded ONNX model live in hw2-vector-search/
sys.path.insert(0, "../hw2-vector-search")
from embedder import Embedder

# llm_structured / calc_price live in evaluation_utils.py, downloaded from the course repo
from evaluation_utils import llm_structured

def _load_api_key_from_env_file(env_file: Path) -> str | None:
    """Parse OPENAI_API_KEY from a .env file. Returns None if not found."""
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line.startswith("OPENAI_API_KEY=") and not line.startswith("#"):
            return line.split("=", 1)[1].strip()
    return None


if not os.environ.get("OPENAI_API_KEY"):
    try:
        os.environ["OPENAI_API_KEY"] = subprocess.check_output(
            ["op", "read", "op://Personal/OpenAIDataTalk/credential"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        # Fallback: read from .env file in the project root
        env_file = Path("../.env")
        key = _load_api_key_from_env_file(env_file) if env_file.exists() else None
        if key:
            os.environ["OPENAI_API_KEY"] = key
        else:
            raise EnvironmentError(
                "OPENAI_API_KEY not found. "
                "Either run 1Password ('op signin') or create a ../.env file with OPENAI_API_KEY=<your-key>"
            )

load_dotenv()
openai_client = OpenAI()

## Loading and chunking the data

Identical to HW2: same 72 lesson pages, same `size=2000, step=1000` chunking → 295 chunks.

In [4]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(documents)} lesson pages")

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks: {len(chunks)}")

Loaded 72 lesson pages
Total chunks: 295


## Rebuilding search from HW2

We rebuild the keyword index (`minsearch.Index`) and the vector index (`minsearch.VectorSearch`) over the chunks, then wrap each in a function with a uniform `(query, num_results=5)` signature — so any evaluation code downstream can swap one search method for another without changes.

In [5]:
embedder = Embedder(path="../hw2-vector-search/models/Xenova/all-MiniLM-L6-v2")

print("Encoding all chunks (this takes a moment)...")
X = np.array(embedder.encode_batch([c["content"] for c in chunks]))
print(f"Embedding matrix shape: {X.shape}")

ki = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
ki.fit(chunks)

vs = minsearch.VectorSearch(keyword_fields=["filename"])
vs.fit(X, chunks)


def text_search(query, num_results=5):
    return ki.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    v = embedder.encode(query)
    return vs.search(v, num_results=num_results)

Encoding all chunks (this takes a moment)...
Embedding matrix shape: (295, 384)


For hybrid search, we reuse the RRF (Reciprocal Rank Fusion) function from HW2: it combines the keyword and vector result lists by rank position (not raw score), so a document that ranks well in *both* lists wins even if it's not #1 in either.

In [6]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

## Q1. Generating ground truth questions

To evaluate search, we need a dataset of questions paired with the document that should answer each one. We generate these with an LLM using **structured output**: instead of asking for a paragraph and parsing it, we ask the model to return a Python object matching a schema we define (a Pydantic model). The model always returns that exact shape — no manual text parsing.

We ask for 5 questions per lesson page, phrased the way a real student would ask online — deliberately *not* copying the page's own wording, so search evaluation stays realistic.

Generating this for all 72 pages costs money and takes time, so Q1 only generates for 3 pages, to measure the average input token cost per call.

In [7]:
class Questions(BaseModel):
    questions: list[str]


data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

q1_filenames = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

input_tokens_list = []

for fname in q1_filenames:
    doc = next(d for d in documents if d["filename"] == fname)
    user_prompt = json.dumps(doc)

    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
    )

    print(f"{fname}")
    print(f"  input_tokens: {usage.input_tokens}")
    print(f"  sample question: {result.questions[0]}")

    input_tokens_list.append(usage.input_tokens)

avg_input_tokens = sum(input_tokens_list) / len(input_tokens_list)
print(f"\nAverage input tokens across 3 calls: {avg_input_tokens:.1f}")

01-agentic-rag/lessons/01-intro.md
  input_tokens: 1020
  sample question: What is Retrieval-Augmented Generation, and why does it help with LLMs’ limits like stale knowledge or missing access to private data?
01-agentic-rag/lessons/02-environment.md
  input_tokens: 1286
  sample question: What do I need installed before starting this module, besides Python and Jupyter?
01-agentic-rag/lessons/03-rag.md
  input_tokens: 1753
  sample question: Why doesn’t a plain LLM give a good answer when I ask it something about a specific course or FAQ?

Average input tokens across 3 calls: 1353.0


**Answer Q1: `1353`**

The average lands around 1350 input tokens per call — each request sends the full page content as JSON plus the instructions, easily a thousand-plus tokens. That's clearly in the 1400 order of magnitude, far from both 140 and 14000.

## The full ground truth

Generating questions for all 72 pages ourselves is out of scope for this homework — the course already did it the same way and shared the result: 360 questions covering every page. We download it instead of regenerating it.

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
wget ${PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv
```

In [8]:
df_ground_truth = pd.read_csv("data/ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
print(f"Ground truth records: {len(ground_truth)}")
ground_truth[0]

Ground truth records: 360


{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

## Q2/Q3. First result: text search vs vector search

Take the first ground truth question and compare which page each search method puts on top.

In [9]:
q = ground_truth[0]["question"]
correct_filename = ground_truth[0]["filename"]
print(f"Question: {q}")
print(f"Correct filename: {correct_filename}")

text_top1 = text_search(q)[0]["filename"]
vector_top1 = vector_search(q)[0]["filename"]

print(f"\ntext_search   top-1: {text_top1}")
print(f"vector_search top-1: {vector_top1}")

Question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
Correct filename: 01-agentic-rag/lessons/01-intro.md

text_search   top-1: 01-agentic-rag/lessons/03-rag.md
vector_search top-1: 01-agentic-rag/lessons/01-intro.md


**Answer Q2: `01-agentic-rag/lessons/03-rag.md`**

**Answer Q3: `01-agentic-rag/lessons/01-intro.md`**

The question was generated from `01-intro.md`. Vector search finds the correct page at the top; keyword search doesn't. One query alone doesn't tell us which method is better overall — that's exactly why we measure across the whole dataset next, instead of trusting a single example.

## Evaluation metrics

We turn search results into two numbers computed across the *whole* ground truth dataset:

- **Hit Rate** — the fraction of questions where the correct page appears anywhere in the top results. Simple recall: did we find it at all?
- **MRR (Mean Reciprocal Rank)** — averages `1 / rank` of the first correct result (0 if not found). It rewards finding the right page *near the top*, not just somewhere in the list. Hit Rate is always an upper bound for MRR.

A result counts as relevant when its `filename` matches the ground truth question's `filename` — that's the only difference from the course's version, which matches on FAQ record `id` instead.

In [10]:
def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(q["question"])
    return [int(d["filename"] == filename) for d in results]


def compute_relevance_total(ground_truth, search_function):
    return [compute_relevance(q, search_function) for q in tqdm(ground_truth)]


def hit_rate(relevance_total):
    return sum(1 for line in relevance_total if 1 in line) / len(relevance_total)


def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance_total)


def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

## Q4. Evaluating text search

In [11]:
text_metrics = evaluate(ground_truth, text_search)
text_metrics

100%|██████████| 360/360 [00:00<00:00, 1097.13it/s]


{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

**Answer Q4: `0.76`**

Hit Rate ≈ 0.758 — keyword search finds the correct page somewhere in the top 5 for about three-quarters of the questions.

## Q5. Evaluating vector search

The module only evaluated keyword search in the lesson — vector search was left for the homework.

In [12]:
vector_metrics = evaluate(ground_truth, vector_search)
vector_metrics

100%|██████████| 360/360 [00:01<00:00, 266.32it/s]


{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

**Answer Q5: `0.55`**

MRR ≈ 0.549. Interestingly, vector search has a *lower* Hit Rate than text search here (0.725 vs 0.758) — keyword matching still wins on raw recall for this dataset, even though vector search can catch semantically-related pages that share no words with the question (as we saw with Q2/Q3).

## Q6. Tuning hybrid search

The RRF constant `k` controls how much top ranks matter: a smaller `k` sharpens the gap between positions, so being #1 counts for a lot more relative to #2 or #3. We sweep `k` over the full ground truth to find the best value empirically, instead of trusting the RRF paper's default of 60.

In [13]:
k_results = []

for k in [1, 50, 100, 200]:
    result = evaluate(
        ground_truth,
        lambda query, k=k: hybrid_search(query, k=k)
    )
    print(f"k={k}: {result}")
    k_results.append({"k": k, **result})

df_k_results = pd.DataFrame(k_results)
df_k_results

100%|██████████| 360/360 [00:01<00:00, 184.68it/s]


k=1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}


100%|██████████| 360/360 [00:01<00:00, 195.64it/s]


k=50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


100%|██████████| 360/360 [00:01<00:00, 210.29it/s]


k=100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


100%|██████████| 360/360 [00:01<00:00, 231.14it/s]


k=200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


,k,hit_rate,mrr
0,1,0.838889,0.648194
1,50,0.836111,0.637917
2,100,0.836111,0.637917
3,200,0.836111,0.637917


In [14]:
# Best MRR; on a tie, prefer the smallest k
best_row = df_k_results.sort_values(["mrr", "k"], ascending=[False, True]).iloc[0]
print(f"Best k: {int(best_row['k'])} (mrr={best_row['mrr']:.4f}, hit_rate={best_row['hit_rate']:.4f})")

Best k: 1 (mrr=0.6482, hit_rate=0.8389)


**Answer Q6: `k=1`**

`k=1` gives the best MRR (≈0.648), clearly ahead of k=50/100/200 (which tie at ≈0.638). A small `k` sharpens the reward for ranking #1, and on this dataset the top rank from each individual method is trustworthy enough that emphasizing it pays off — the opposite of assuming the RRF paper's default of 60 is automatically right for our data.

## Summary of answers

| Question | Answer |
|---|---|
| Q1. Average input tokens (3 pages) | `1400` |
| Q2. `text_search` top-1 filename for `ground_truth[0]` | `01-agentic-rag/lessons/03-rag.md` |
| Q3. `vector_search` top-1 filename for `ground_truth[0]` | `01-agentic-rag/lessons/01-intro.md` |
| Q4. Hit Rate of `text_search` | `0.76` |
| Q5. MRR of `vector_search` | `0.55` |
| Q6. Best `k` for `hybrid_search` (RRF) | `1` |

Submit at: https://courses.datatalks.club/llm-zoomcamp-2026/homework/hw4